# Day 4 — Workflow patterns: chain, route, parallelize, orchestrate, evaluate
**Theory:** [day-04-workflow-patterns.md](../theory/day-04-workflow-patterns.md) · ⏱ ~60 min · Level: easy → medium

Most production "agents" are compositions of these 5 patterns. We'll implement each on support tickets and
measure cost and latency.

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone + install the course repo, read the key from notebook secrets
    if not pathlib.Path("agentic-ai-course").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Rahulverma5/agentic-ai-course"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "agentic-ai-course/requirements.txt"], check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("GEMINI_API_KEY", userdata.get("GEMINI_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("GEMINI_API_KEY", UserSecretsClient().get_secret("GEMINI_API_KEY"))
    except Exception as e:
        print("Add GEMINI_API_KEY to the notebook secrets:", e)
    REPO = pathlib.Path("agentic-ai-course").resolve()
else:  # Codespaces / local: find the repo root
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
from google.genai import types  # Gemini SDK types: Content, Part, GenerateContentConfig, ...
from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of, thinking, user
from shopease.data import setup_data
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup();

In [ ]:
import time, concurrent.futures as cf
import pandas as pd

TICKETS = [
    "My smart watch O2001 stopped working after a week. I want my money back.",
    "How long does standard shipping take?",
    "Order O2003 still hasn't arrived and it's been 18 days!!!",
    "Do gold members get free returns?",
    "I'm going to sue you if nobody answers me today.",
    "Can I cancel order O2005?",
    "What's the warranty on smart watches?",
    "The earbuds from O2006 were great, thanks!",
]
results = []
def timed(pattern, fn, *args):
    t0, c0 = time.time(), tracker.total()
    out = fn(*args)
    results.append({"pattern": pattern, "seconds": round(time.time() - t0, 2), "cost $": round(tracker.total() - c0, 5)})
    return out

## 1. Prompt chaining with a code gate

In [ ]:
def chain(ticket):
    summary = ask(f"Summarize this support ticket in ONE line:\n{ticket}")
    if len(summary) > 200:                                   # gate: plain code, not an LLM
        summary = ask(f"Shorten to under 20 words:\n{summary}")
    return ask(f"Write a 2-sentence polite reply to a customer whose issue is: {summary}")

print(timed("chaining", chain, TICKETS[0]))

## 2. Routing: a cheap classifier sends traffic to the right path

In [ ]:
from typing import Literal
from pydantic import BaseModel

class Route(BaseModel):
    route: Literal["faq", "order_issue", "human"]
    reason: str

def route(msg):
    r = client.models.generate_content(model=CHEAP_MODEL, contents=msg, config=types.GenerateContentConfig(
        system_instruction="Classify ShopEase support messages. faq = general policy question; order_issue = needs "
                           "account/order data; human = legal threats, abuse or asks for a person.",
        response_mime_type="application/json", response_schema=Route))
    tracker.add(CHEAP_MODEL, r.usage_metadata, "router")
    return r.parsed

routes = [timed("routing", route, t) for t in TICKETS]
df = pd.DataFrame({"ticket": TICKETS, "route": [r.route for r in routes]})
print(df.route.value_counts().to_dict())
df

**Why this saves money:** only `order_issue` traffic needs the expensive agent. Estimate the saving if an agent
run costs ~$0.05 and the router ~$0.0002:

In [ ]:
n_agent = (df.route == "order_issue").sum()
print(f"without router: {len(df)} × $0.05 = ${len(df) * 0.05:.2f}  (illustrative paid-tier prices)")
print(f"with router:    {len(df)} × $0.0002 + {n_agent} × $0.05 + {len(df) - n_agent} × $0.002 = "
      f"${len(df) * 0.0002 + n_agent * 0.05 + (len(df) - n_agent) * 0.002:.2f}")

## 3. Parallelization: independent checks at the same time

In [ ]:
CHECKS = {"polite": "Is this reply polite? Answer yes or no.",
          "no_promises": "Does this reply avoid promising a refund without checking eligibility? Answer yes or no.",
          "concise": "Is this reply under 60 words? Answer yes or no."}
def review(reply):
    with cf.ThreadPoolExecutor(max_workers=3) as pool:
        futs = {k: pool.submit(ask, f"{q}\n\nReply:\n{reply}", max_tokens=20) for k, q in CHECKS.items()}
    return {k: f.result().strip().lower().startswith("yes") for k, f in futs.items()}

draft = "Hi! Sure, I've refunded your watch already, money is on its way!"
print(timed("parallel (3 checks)", review, draft))

**Voting math:** 3 independent judges with accuracy p, majority vote → p³ + 3p²(1−p).

In [ ]:
for p in (0.7, 0.8, 0.9):
    print(f"p={p}: majority of 3 = {p**3 + 3 * p**2 * (1 - p):.3f}")

## 4. Orchestrator–workers: decompose at runtime

In [ ]:
class Plan(BaseModel):
    subtasks: list[str]

def orchestrate(request):
    plan = client.models.generate_content(
        model=MODEL, contents=f"Split this into 2-4 independent writing subtasks:\n{request}",
        config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=Plan,
                                           thinking_config=thinking("low")))
    tracker.add(MODEL, plan.usage_metadata, "orchestrator")
    with cf.ThreadPoolExecutor(max_workers=2) as pool:  # small pool: free-tier RPM
        parts = list(pool.map(lambda s: ask(f"Do this in under 60 words: {s}"), plan.parsed.subtasks))
    return ask("Combine these parts into one short FAQ section:\n\n" + "\n\n".join(parts), max_tokens=800)

print(timed("orchestrator-workers", orchestrate, "Write FAQ answers about ShopEase returns, shipping and warranty."))

## 5. Evaluator–optimizer: write, critique, revise

In [ ]:
def write_with_review(task, max_rounds=3):
    draft = ask(task)
    for i in range(max_rounds):
        verdict = ask("Criteria: polite, under 80 words, mentions the 30-day return window. Reply PASS, or list fixes."
                      f"\n\nDraft:\n{draft}")
        print(f"round {i + 1}: {verdict.strip()[:80]}")
        if verdict.strip().upper().startswith("PASS"):
            break
        draft = ask(f"Revise the draft using this feedback:\n{verdict}\n\nDraft:\n{draft}")
    return draft

print(timed("evaluator-optimizer", write_with_review, "Reply to a customer asking if they can return shoes they bought 2 weeks ago."))

## 6. Compare the patterns

In [ ]:
pd.DataFrame(results).groupby("pattern").agg(calls=("seconds", "size"), seconds=("seconds", "sum"), cost=("cost $", "sum"))

## ✏️ Exercises
1. Write 10 more tickets (include tricky ones) and measure the router's accuracy against your own labels.
2. In `review`, use different prompts per judge for the same check and majority-vote. Does it catch the bad draft more reliably?
3. Design (on paper) the pattern combination for a "weekly sales report" product.

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker` (estimated at paid prices; the free tier costs $0). Make a habit of checking tokens and cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None